# Ops1 — Index

1. [Filtering - select, Normalization & Conditional Columns (CASE WHEN / F.when)](#command/ec10554e-e7c5-48b8-854b-62d172aa3f60)
2. [Standard distinct, Aggregations, Conditional GROUP BY & FILTER (WHERE)](#command/576e0ec3-853b-4238-bc7c-24237b5b49e1)
3. [ROLLUP, CUBE, GROUPING SETS & Advanced Aggregates](#command/793c3a24-872b-4222-8b2f-9616da514d1d)
4. [Join Types & Anti-Joins](#command/63d277f9-a0ff-4534-87ad-27c9d47eb701)
5. [Recursive and other ops](#command/9559f9e1-e88a-4dc8-932f-e3fc738323f3)
6. [Complete 16-Query Window Cheat Sheet](#command/03887365-2a79-4cf7-9724-1bb7c6c70a88)
7. [20 Must-Know Interview Patterns](#command/c4f4ef9b-741e-4174-a399-c8469079fcf8)

In [0]:
from pyspark.sql import functions as F

### Filtering - select, Normalization & Conditional Columns (CASE WHEN / F.when)

In [0]:
sales_df = spark.table("sql_master.sql_master_pack.sales")

print(f"sales_df columns: {', '.join(sales_df.columns)}")

print("sales_df schema:")
sales_df.printSchema()

row_count = sales_df.count()
print(f"sales_df has {row_count:,} rows")       #:, is for readability - sales_df has 1,500,000 rows

sales_df.limit(4).display()

In [0]:
classified = sales_df.select(
    "sale_id",
    "amount",
    F.when(F.col("amount") < 100, "small")
     .when(F.col("amount") <= 200, "medium")
     .otherwise("large")
     .alias("sale_size")   
).persist()  # Cache in memory/disk to prevent re-computing on the next action
#classified.show()

classified.display()

# Always unpersist when finished
classified.unpersist()

### Standard distinct, Aggregations, Conditional GROUP BY & FILTER (WHERE)

In [0]:
unique_products_df = sales_df.select("product_id").distinct()
unique_products_df.show()

#deduped_df = sales_df.dropDuplicates(["product_id"])

In [0]:
sales_df.groupBy("product_id").agg(
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.count("*").alias("transaction_count"),
).show()

### ROLLUP, CUBE, GROUPING SETS & Advanced Aggregates

In [0]:
# Rollup + GROUPING()
sales_df.rollup("region", "product_id").agg(
    F.sum("amount").alias("total"),
    F.grouping("region").alias("region_is_total"),
    F.grouping("product_id").alias("product_is_total"),
).orderBy("region", "product_id").show()

# Cube + Advanced Aggregates
sales_df.cube("region", "product_id").agg(F.sum("amount").alias("total")).show()

sales_df.groupBy("region").agg(
    F.collect_set("product_id").alias("products"),
    F.count_distinct("product_id").alias("n_products"),
    F.approx_count_distinct("sale_id", rsd=0.02).alias("approx_sales"),
    F.percentile_approx("amount", [0.5, 0.9], 10000).alias("p50_p90"),
    F.first("amount", ignorenulls=True).alias("first_amount"),
).show()

### Join Types & Anti-Joins

| Join Type | Keeps |
| :--- | :--- |
| `inner` | Matching rows from both sides |
| `left` / `right` / `full` | All left / right / both sides |
| `left_semi` | Left rows with a match (no right columns) |
| `left_anti` | Left rows with no match (cleaner & safer than `NOT IN`) |

In [0]:
cust_df = spark.table("sql_master.sql_master_pack.customers")
ord_df = spark.table("sql_master.sql_master_pack.orders")

print(f"cust_df columns: {', '.join(cust_df.columns)}")

unique_cust_df = cust_df.select("cust_id").distinct().orderBy("cust_id")

unique_cust_df.display()

In [0]:


inner_j = cust_df.join(ord_df, "cust_id", "inner") # Keeps only matching rows between both DataFrames
left_j  = cust_df.join(ord_df, "cust_id", "left")  # Keeps all rows from cust_df and matching rows from ord_df
left_sj  = cust_df.join(ord_df, "cust_id", "left_semi")  #Keeps rows from cust_df that have AT LEAST ONE matching order in ord_df.
anti_j  = cust_df.join(ord_df, "cust_id", "left_anti") # Customers with 0 orders   Keeps rows from cust_df that have NO matching order in ord_df

full_j = cust_df.join(ord_df, "cust_id", "full")   # Keeps all customers and all orders, with NULL fills where either is missing (useful for reconciliation)

right_j = cust_df.join(ord_df, "cust_id", "right") # Keeps all orders and matching customers (identifies orphaned orders placed with non-existent cust_ids).

inner_j = ord_df.join(F.broadcast(cust_df), "cust_id", "inner") # If one of the tables is small (e.g., cust_df is a small dimension table or lookup under ~10MB–100MB), you should advise applying a broadcast hint to avoid an expensive network shuffle (SortMergeJoin / ShuffleHashJoin):


# Join Fan-Out Check (Always validate dimension key uniqueness before joining!)
dupes = cust_df.groupBy("cust_id").count().filter(F.col("count") > 1)
assert dupes.count() == 0, "Dimension key is not unique!"

In [0]:
left_sj.display()

In [0]:
anti_j.display()

### Recursive and other ops

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

emp_df = spark.table("sql_master.sql_master_pack.employees")

print(f"emp_df columns: {', '.join(emp_df.columns)}")

row_count = emp_df.count()
print(f"emp_df has {row_count:,} rows")       #:, is for readability - sales_df has 1,500,000 rows

In [0]:
# Scalar & Correlated Subqueries

# SELECT first_name, last_name, salary
# FROM employees
# WHERE salary > (SELECT AVG(salary) FROM employees);

# avg_emp_sal_df = emp_df.select(F.avg("salary")).first()[0]

# above_company_avg = emp_df.filter(F.col("salary") > avg_emp_sal_df).select ("first_name", "last_name", "salary")
# above_company_avg.show()


above_avg = (
    emp_df
    .withColumn("avg_emp_sal", F.avg("salary").over(Window.partitionBy()))
    .filter(F.col("salary") > F.col("avg_emp_sal"))
    .select("first_name", "last_name", "salary")
)

above_avg.show()

In [0]:
# EXISTS (Safer than IN when NULLs may exist)

# SELECT e1.first_name, e1.last_name, e1.dept_id, e1.salary
# FROM employees e1
# WHERE e1.salary > (
#     SELECT AVG(e2.salary) FROM employees e2 WHERE e2.dept_id = e1.dept_id
# );

#dept_window = Window.partitionBy("dept_id")

above_dept_avg = (
    emp_df
    .withColumn("dept_avg", F.avg("salary").over(Window.partitionBy("dept_id")))
    .filter(F.col("salary") > F.col("dept_avg"))
    .select("first_name", "last_name", "dept_id", "salary","dept_avg")
)

above_dept_avg.show()

In [0]:
# SELECT e1.first_name, e1.last_name, e1.dept_id
# FROM employees e1
# WHERE EXISTS (
#     SELECT 1 FROM employees e2 WHERE e2.manager_id = 110 AND e2.dept_id = e1.dept_id
# );

# The direct mathematical equivalent of WHERE EXISTS in PySpark is a left_semi join. It filters the left DataFrame against matches in the right DataFrame without duplicating rows or appending right-side columns from the right DataFrame

# Target departments managed by employee 110
mgr_depts = (
    emp_df
    .filter(F.col("manager_id") == 110)
    .select("dept_id")
    .distinct()
)

# left_semi acts as WHERE EXISTS
exists_df = (
    emp_df.alias("e1")
    .join(mgr_depts, on="dept_id", how="left_semi")
    .select("first_name", "last_name", "dept_id")
)

exists_df.show()

In [0]:
subordinates_df = spark.sql("""
    WITH RECURSIVE subordinate AS (
        SELECT employee_id, first_name, last_name, manager_id,
               0 AS hierarchy_level,
               CONCAT(first_name, ' ', last_name) AS reporting_path
        FROM sql_master.sql_master_pack.employees
        WHERE employee_id = 110
        UNION ALL
        SELECT e.employee_id, e.first_name, e.last_name, e.manager_id,
               s.hierarchy_level + 1,
               CONCAT(s.reporting_path, ' -> ', e.first_name, ' ', e.last_name)
        FROM sql_master.sql_master_pack.employees e
        JOIN subordinate s ON e.manager_id = s.employee_id
    )
    SELECT * FROM subordinate ORDER BY hierarchy_level, employee_id
""")

subordinates_df.display()

In [0]:
# 1. Base / Anchor Case (Level 0)
current_level = (
    emp_df
    .filter(F.col("employee_id") == 110)
    .select(
        "employee_id",
        "first_name",
        "last_name",
        "manager_id",
        F.lit(0).alias("hierarchy_level"),
        F.concat_ws(" ", "first_name", "last_name").alias("reporting_path")
    )
)

accumulated = current_level

# 2. Iterative recursive loop
while True:
    next_level = (
        emp_df.alias("e")
        .join(current_level.alias("s"), F.col("e.manager_id") == F.col("s.employee_id"), "inner")
        .select(
            F.col("e.employee_id"),
            F.col("e.first_name"),
            F.col("e.last_name"),
            F.col("e.manager_id"),
            (F.col("s.hierarchy_level") + 1).alias("hierarchy_level"),
            F.concat(
                F.col("s.reporting_path"),
                F.lit(" -> "),
                F.concat_ws(" ", F.col("e.first_name"), F.col("e.last_name"))
            ).alias("reporting_path")
        )
    )

    # Stop when no deeper subordinates exist
    if next_level.isEmpty():
        break

    # Checkpoint/persist can be used here if the hierarchy is deeply nested (> 10 levels)
    accumulated = accumulated.unionByName(next_level)
    current_level = next_level

result_org_chart = accumulated.orderBy("hierarchy_level", "employee_id")

result_org_chart.display()


### Complete 16-Query Window Cheat Sheet

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

emp_sal = spark.table("sql_master.sql_master_pack.employee_salary")
w_sal = Window.partitionBy("Department").orderBy(F.col("Salary").desc(), F.col("EmpID"))
w_dt  = Window.partitionBy("Department").orderBy("JoiningDate", "EmpID")
w_full = w_sal.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
w_part = Window.partitionBy("Department")

emp_sal.select(
    "EmpID", "EmpName", "Department", "Salary", "JoiningDate",
    F.row_number().over(w_sal).alias("rn"),
    F.rank().over(w_sal).alias("rnk"),
    F.dense_rank().over(w_sal).alias("drnk"),
    F.percent_rank().over(w_sal).alias("pct_rank"),
    F.cume_dist().over(Window.orderBy(F.col("Salary").desc())).alias("cume_dist"),
    F.ntile(4).over(Window.orderBy(F.col("Salary").desc())).alias("Quartile"),
    F.lag("Salary", 1).over(w_dt).alias("PrevSalary"),
    F.lead("Salary", 1).over(w_dt).alias("NextSalary"),
    F.first("Salary").over(w_full).alias("HighestSalary"),
    F.last("Salary").over(w_full).alias("LowestSalary"),
    F.nth_value("Salary", 2).over(w_full).alias("SecondHighestSalary"),
    F.sum("Salary").over(w_dt).alias("RunningTotal"),
    F.round(F.avg("Salary").over(w_dt), 2).alias("RunningAverage"),
    F.min("Salary").over(w_part).alias("MinSalary"),
    F.max("Salary").over(w_part).alias("MaxSalary"),
    F.count("*").over(w_part).alias("DeptCount"),
    F.round(F.col("Salary") * 100.0 / F.sum("Salary").over(w_part), 2).alias("SalaryPct")
).show()

# 16. Top 1 employee per department (deterministic tiebreaker)
top1 = emp_sal.withColumn("rn", F.row_number().over(w_sal)).filter(F.col("rn") == 1).drop("rn")
top1.show()

### 20 Must-Know Interview Patterns

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

order_df = spark.table("sql_master.sql_master_pack.orders")
emp_sal_df = spark.table("sql_master.sql_master_pack.employee_salary")
cloud_spend_df = spark.table("sql_master.sql_master_pack.cloud_spend")
user_logins_df = spark.table("sql_master.sql_master_pack.user_logins")
customers_df = spark.table("sql_master.sql_master_pack.customers")
orders_df = spark.table("sql_master.sql_master_pack.orders")
employees_df = spark.table("sql_master.sql_master_pack.employees")
df = spark.table('information_schema.tables').filter("table_schema = 'sql_master_pack'").select('table_name')
display(df)

In [0]:
# 1. Nth highest salary (N = 2) using dense_rank without partition
res_1 = (
    emp_sal_df
    .withColumn("rk", F.dense_rank().over(Window.orderBy(F.col("Salary").desc())))
    .filter(F.col("rk") == 2)
    .drop("rk")
)

res_1.display()

In [0]:
# 2. Deduplicate / Get latest order status by partitioning on order_id
res_2 = (
    order_df
    .withColumn("rn", F.row_number().over(Window.partitionBy("order_id").orderBy(F.col("order_dt").desc(), F.col("status").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

res_2.display()


In [0]:
# 3. Top 3 earners per department sorted by salary descending and EmpID tiebreaker
res_3 = (
    emp_sal_df
    .withColumn("rn", F.row_number().over(Window.partitionBy("Department").orderBy(F.col("Salary").desc(), F.col("EmpID").asc())))
    .filter(F.col("rn") <= 3)
    .drop("rn")
)

res_3.display()

In [0]:
#4. Running total of sales ordered cumulatively by order date and order_id

res_4 = (
    order_df
    .select(
        "order_id", "order_dt", "amount",
        F.sum("amount").over(Window.orderBy("order_dt", "order_id")).alias("running_total")
    )
)

res_4.display()

In [0]:
# 5. Month-over-month spend change by aggregating by month and applying lag()
m = (
    cloud_spend_df
    .groupBy(F.date_trunc("month", "spend_month").alias("mo"))
    .agg(F.sum("cost_usd").alias("amt"))
)

res_5 = (
    m.select(
        "mo", "amt",
        (F.col("amt") - F.lag("amt", 1).over(Window.orderBy("mo"))).alias("mom_change")
    )
)

res_5.display()

In [0]:
# 6. Consecutive login streaks grouped by the difference between date and row_number
distinct_days = user_logins_df.select("user_id", "login_date").distinct()
numbered = (
    distinct_days
    .withColumn("rn", F.row_number().over(Window.partitionBy("user_id").orderBy("login_date")))
    .withColumn("grp", F.date_sub(F.col("login_date"), F.col("rn")))
)
res_6 = (
    numbered
    .groupBy("user_id", "grp")
    .agg(
        F.min("login_date").alias("start_dt"),
        F.max("login_date").alias("end_dt"),
        F.count("*").alias("consecutive_days")
    )
    .drop("grp")
    .orderBy("start_dt")
)

res_6.display()

In [0]:
# 7. Customers with no orders using a left-anti join
res_7 = customers_df.join(orders_df, on="cust_id", how="left_anti")

res_7.display()

In [0]:
# 8. Find duplicate order IDs having a count greater than 1
res_8 = (
    orders_df
    .groupBy("order_id")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)

res_8.display()

In [0]:
# 9. Employees earning more than their manager via self-join
res_9 = (
    employees_df.alias("e")
    .join(employees_df.alias("m"), F.col("e.manager_id") == F.col("m.employee_id"), "inner")
    .filter(F.col("e.salary") > F.col("m.salary"))
    .select(
        F.concat_ws(" ", "e.first_name", "e.last_name").alias("emp"),
        F.col("e.salary").alias("emp_salary"),
        F.concat_ws(" ", "m.first_name", "m.last_name").alias("mgr"),
        F.col("m.salary").alias("mgr_salary")
    )
)

res_9.display()


In [0]:
# 10. Pivot quarterly months into columns using conditional aggregation
res_10 = (
    cloud_spend_df
    .groupBy("environment")
    .agg(
        F.sum(F.when(F.month("spend_month") == 1, F.col("cost_usd")).otherwise(0)).alias("Jan"),
        F.sum(F.when(F.month("spend_month") == 2, F.col("cost_usd")).otherwise(0)).alias("Feb"),
        F.sum(F.when(F.month("spend_month") == 3, F.col("cost_usd")).otherwise(0)).alias("Mar")
    )
)

res_10.display()

In [0]:
# 11. Earliest purchase date per customer using simple aggregation
res_11 = (
    orders_df
    .groupBy("cust_id")
    .agg(F.min("order_dt").alias("first_purchase_dt"))
)

res_11.display()

In [0]:
# 12. Second order per user by ranking orders and filtering on row_number = 2
res_12 = (
    orders_df
    .withColumn("rn", F.row_number().over(Window.partitionBy("cust_id").orderBy("order_dt", "order_id")))
    .filter(F.col("rn") == 2)
    .drop("rn")
)

res_12.display()

In [0]:
# 13. Median salary per department computed via percentile_approx (or median)
res_13 = (
    emp_sal_df
    .groupBy("Department")
    .agg(F.percentile_approx("Salary", 0.5).alias("median_salary"))
)

res_13.display()

In [0]:
# 14. Customers who purchased both products 'A' and 'B' using filtered distinct count
res_14 = (
    orders_df
    .filter(F.col("product").isin("A", "B"))
    .groupBy("cust_id")
    .agg(F.count_distinct("product").alias("p_cnt"))
    .filter(F.col("p_cnt") == 2)
    .select("cust_id")
)

res_14.display()

In [0]:
# 15. Identify sequence gaps by finding where the next ID exceeds the current ID by > 1
seq = (
    orders_df
    .select("order_id").distinct()
    .withColumn("next_id", F.lead("order_id", 1).over(Window.orderBy("order_id")))
)
res_15 = (
    seq
    .filter((F.col("next_id") - F.col("order_id")) > 1)
    .select(
        F.col("order_id").alias("gap_after"),
        F.col("next_id").alias("gap_before"),
        (F.col("next_id") - F.col("order_id") - 1).alias("missing_ids")
    )
)

res_15.display()

In [0]:
# 16. Employee salary percentage relative to the overall total payroll
res_16 = (
    emp_sal_df
    .select(
        "EmpName", "Salary",
        F.round((F.col("Salary") * 100.0) / F.sum("Salary").over(Window.partitionBy()), 2).alias("pct_of_total")
    )
)

res_16.display()

In [0]:
# 17. Null count comparison between total row count and populated email rows
res_17 = (
    customers_df
    .select(
        F.count("*").alias("total_rows"),
        F.count("email").alias("non_nulls"),
        (F.count("*") - F.count("email")).alias("null_count")
    )
)

res_17.display()


In [0]:
# 19. Customers who purchased only one unique product type
res_19 = (
    orders_df
    .groupBy("cust_id")
    .agg(
        F.count_distinct("product").alias("distinct_cnt"),
        F.min("product").alias("only_product")
    )
    .filter(F.col("distinct_cnt") == 1)
    .select("cust_id", "only_product")
)

res_19.display()

In [0]:
# 20. Year-over-year spend comparison for Prod compute using month/year self-join
mp = (
    cloud_spend_df
    .filter((F.col("environment") == "Prod") & (F.col("service") == "Compute"))
    .groupBy(F.year("spend_month").alias("yr"), F.month("spend_month").alias("mo"))
    .agg(F.sum("cost_usd").alias("amt"))
)
res_20 = (
    mp.alias("a")
    .join(mp.alias("b"), (F.col("a.mo") == F.col("b.mo")) & (F.col("a.yr") == F.col("b.yr") + 1), "left")
    .select(
        F.col("a.yr"),
        F.col("a.mo"),
        F.col("a.amt").alias("cur_amt"),
        F.col("b.amt").alias("prev_yr_amt"),
        (F.col("a.amt") - F.col("b.amt")).alias("yoy_diff")
    )
)

res_20.display()